# From molecular structures to predictions
## Python, RDKit and introductory machine learning


**Question:** can molecular structure help us predict aqueous solubility?

You will read and draw molecules, calculate descriptors, explore data, train regression models, evaluate predictions, . We will also ask which conclusions the data can support.

**Resources:** keep `delaney-processed.csv` in the same folder as this notebook.

In this workshop you will use a package called RDKit, rdkit is a commonly used library in data science and cheminformatics. Cheminformatics is an *in silico* technique that is used to index and analyse data related to the physical principles underpinning the properties of chemicals. Cheminformatics is used across a wide range of fields for example, to aid and inform the process of drug discovery, in the design of well-defined combinatorial libraries of synthetic compounds, or to assist in materials discovery and optimisation. 

Frank Brown (Associate VP Merck chemicals) was an early proponent of cheminformatics who coined the term Cheminformatics, in 1998 he was quoted as saying:

&emsp;*"Chemoinformatics is the mixing of those information resources to transform data into information and information into knowledge for the intended purpose of making better decisions faster"*

In this workshop you will learn how to combine machine learning (sklearn) with the Cheminformatics in rdkit in order to make sensible predictions about chemical properties.

### Task 1 — Install rdkit

Open the terminal in VSCode and install rdkit and sklearn using the commands, or using pip3 depending on your install:

> pip install rdkit\
> pip install scikit-learn

The other packages imported below are already installed.

Now run the cell below to import the relevant packages.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rdkit
import sklearn
import seaborn as sns
from rdkit import Chem, DataStructs
from rdkit.Chem import Descriptors, Draw, rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold
from IPython.display import display
from sklearn.model_selection import train_test_split, KFold, cross_validate, GroupShuffleSplit
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
SEED = 42
print(f'Python: {sys.version.split()[0]} RDKit: {rdkit.__version__} sklearn: {sklearn.__version__}')

### Task 2 - Represent and inspect a molecule
**Cheminformatics** uses computers to represent and analyse chemical information.

**Remember**: SMILES (Simplified Molecular Input Line Entry System) writes a molecular graph as text: atoms are vertices and bonds are connections. `CCO` is ethanol; parentheses indicate branches, digits close rings, and lowercase `c` denotes aromatic carbon. Hydrogens are often implicit. SMILES is not a set of 3D coordinates, and different strings can describe the same molecule.

An RDKit **Mol object** stores a molecular graph. Parsing means interpreting the string as a structure; invalid input can return `None`. The aspirin SMILES below is provided.

In [ ]:
aspirin_smiles = 'CC(=O)Oc1ccccc1C(=O)O'
aspirin = Chem.MolFromSmiles(aspirin_smiles)
if aspirin is None:
    raise ValueError('The SMILES could not be parsed.')
display(Draw.MolToImage(aspirin))
print(f'Explicit atoms: {aspirin.GetNumAtoms()}')
print(f'Bonds: {aspirin.GetNumBonds()}')
for atom in aspirin.GetAtoms():
    print(f"{atom.GetIdx()} {atom.GetSymbol()}")

In the cell below parse, drawn and count the explicit atoms in ethanol, the smiles string is `CCO`.

### Task 3 - Turn chemistry into numbers 
A **descriptor** is a numerical summary of a molecule. A **feature** is an input used by a model; descriptors can serve as features.

| Descriptor | Meaning | Units or convention |
|---|---|---|
| MolWt | Molecular weight calculated from the structure | numerical molar mass in g mol⁻¹ |
| MolLogP | RDKit Crippen estimate of log₁₀ octanol/water partition coefficient | dimensionless; larger values indicate greater lipophilicity |
| TPSA | Topological polar surface area, estimated from atom environments | Å²; no 3D structure required |
| NumHDonors | Count of hydrogen-bond donors under RDKit rules | integer |
| NumHacceptors | Count of hydrogen-bond acceptors under RDKit rules | integer |
| NumRotatableBonds | Count of rotatable bonds under RDKit rules | integer |

LogP describes partitioning of a species, not its measured aqueous solubility. Ionisation, pH, crystal packing and temperature also matter. A molecular graph alone does not encode all of these factors.

The table lists RDKit descriptor function names; the dictionary keys in the next cell define the shorter output column names (for example, `MolLogP` becomes `LogP`).

In the cell below define a function that returns molecular weight, LogP, TPSA, H bond acceptors, H bond donors and Rotational bonds. Then print the pandas series (pd.Series) when the function is called for asprin. A `Series` is a labelled one-dimensional column of values.

In [ ]:
def calculate_descriptors(mol):
    return {
        'MolWt': Descriptors._____(mol),
        'LogP': Descriptors.______(mol),
        'TPSA': Descriptors.______(mol),
        'HBD': Descriptors._______(mol),
        'HBA': Descriptors.________(mol),
        'RotBonds': Descriptors.________(mol),
    }
print(f"{____._____(________(_______))}")

Calculate the same descriptors for ethanol. 

### Task 4 - Load experimental measurements and check data quality 

Before we can train a machine learning model, we need a dataset containing experimental measurements. A **dataset** is a collection of information organised into rows and columns. In this example, each row represents a chemical compound, while the columns contain information about that compound, such as its name, molecular structure and experimentally measured solubility.

We will use the **Delaney (ESOL) dataset**, which contains experimental measurements of aqueous solubility for a range of organic molecules. The dataset is provided as a CSV (*comma-separated values*) file, which we can load into Python using pandas.

Our aim is to develop a machine learning model that predicts **aqueous solubility from molecular structure**. The property we want to predict is called the **target variable**. In this case, it is the experimentally measured solubility, expressed as log S:

$$
\log S = \log_{10}\left(\frac{S}{1\;\mathrm{mol\,L^{-1}}}\right)
$$

Solubility is expressed on a logarithmic scale because different compounds can have very different solubilities. For example, a log S of −3 corresponds to a solubility of 0.001 mol L⁻¹, while a log S of −5 corresponds to 0.00001 mol L⁻¹. A difference of one log unit represents a tenfold change in solubility.

The dataset also contains a column called `ESOL predicted`, containing predictions from an existing model. **We will not use this column**, because we want to train our own model using experimental measurements rather than predictions made by another model.

#### Load and prepare the dataset

In the next code cell, we will prepare the experimental data for machine learning.

1. **Load the dataset:** Use `pd.read_csv()` to read the CSV file into a pandas DataFrame, which is a table containing labelled rows and columns.

2. **Select the relevant information:** Keep the compound name (`Compound ID`), molecular structure (`smiles`) and experimentally measured solubility. The SMILES string is a text representation of the atoms and bonds in a molecule.

3. **Check for missing or invalid data:** Convert the solubility measurements into numerical values using `pd.to_numeric()`. The argument `errors="coerce"` replaces values that cannot be converted with `NaN` (missing values). Remove compounds with missing SMILES or solubility measurements.

4. **Check molecular structures:** Use RDKit's `Chem.MolFromSmiles()` function to convert each SMILES string into a molecular object. Some SMILES strings may be invalid, so we remove any molecules that RDKit cannot interpret.

5. **Identify duplicate molecules:** Different SMILES strings can sometimes represent the same molecule. Use RDKit to generate a *canonical SMILES*, a consistent representation of the molecular structure, so we can identify repeated structures. For this exercise, we will retain only the first occurrence of each structure to avoid including the same molecule multiple times. In research, repeated measurements should be investigated before deciding how to handle them.

6. **Inspect the cleaned dataset:** Display the first five rows and check how many compounds remain after removing missing values, invalid structures and duplicates.

By the end of this exercise, you should have a dataset containing valid molecular structures and experimental solubility measurements, ready for calculating molecular descriptors and training a machine learning model.

**Tips**:`[]` selects columns; `.copy()` creates a separate table. `.head()` displays the first five rows. `.map()` applies a function to each value. A **lambda** is a short unnamed function. `.dropna()` removes missing entries; `errors='coerce'` turns non-numeric target entries into missing values. An **assertion** stops execution if a required condition is false.

In [ ]:
DATA_PATH = Path('delaney-processed.csv')
##Statement to print error message if the data are not found.
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Dataset not found: {DATA_PATH}')
#Read data into pandas dataframe
raw = 
TARGET = 'measured log solubility in mols per litre'
data = raw[['_______', '________', TARGET]].copy()
#Convert data to numeric
data[TARGET] = ______(______[TARGET], errors='coerce')
#remove missing entries
data = _______._____(subset=['smiles', TARGET]).copy()
#Check molecular structure
data['mol'] = data['smiles'].map(_____.______)
invalid = data['mol'].isna().sum()
data = data.loc[data['mol'].notna()].copy()
data['canonical'] = data['mol'].map(lambda m: Chem.MolToSmiles(m, isomericSmiles=True))
print(f'Raw rows: {len(raw)} invalid SMILES: {invalid}')
print(f"Repeated canonical structures: {data['canonical'].duplicated().sum()}")
# Retain the first reported observation for each exact canonical structure.
# A research analysis should inspect repeat measurements and their conditions separately.
data = data.drop_duplicates('canonical', keep='first').reset_index(drop=True)
print(f'Usable unique structures: {len(data)}')
display(data[['Compound ID', 'smiles', TARGET]].head())
assert np.isfinite(data[TARGET]).all()

### Task 5 - Preparing the data for machine learning

We will use supervised machine learning to predict aqueous solubility from molecular structure. The model learns relationships between features (molecular properties such as molecular weight and LogP) and a known target (experimentally measured solubility, log S).

Because we are predicting a numerical value, this is called regression. Predicting molecular properties from chemical structure is known as quantitative structure–property relationship (QSPR) modelling.

**Dividing the dataset**

We need to test whether our model can predict solubility for molecules it has not encountered before. We will therefore divide our dataset into three groups:

- Training set (80% of the molecules remaining after withholding ten): Used to train the model.

- Test set (20% of the molecules remaining after withholding ten): Used to evaluate the model's predictions on unseen molecules.

- Blind challenge (10 molecules): Set aside for a final prediction exercise, without examining their experimental solubilities.

We first reserve 10 molecules, then divide the remaining data into training and test sets using train_test_split(). The argument test_size specifies how much data to reserve, while random_state makes the random selection reproducible.

#### Your task

In the next code cell:

1. Divide the dataset into training, test and blind challenge sets. Use 'train_test_split()' to randomly select 10 molecules for the final blind challenge. Store the rest of the dataset in *remaining* and the 10 selected molecules in *blind*. The function returns two datasets, which we assign to these variables in a single line.

2. Use the calculate_descriptors() function to calculate molecular properties for each molecule.

3. Create X_train and X_test containing the molecular descriptors (features), and y_train and y_test containing experimental solubilities (targets).

4. Display the number of molecules in each set and the first five rows of X_train.

Why should we avoid examining the test and blind challenge data when developing our model? Using information from these datasets to influence model development is called *data leakage* and can lead to misleadingly good results.

In [ ]:
# Reserve 10 molecules for the final blind challenge
remaining, blind = train_test_split(
    data,
    test_size=____,
    random_state=SEED
)

# Divide the remaining molecules into 80% training and 20% testing
train, test = train_test_split(
    remaining,
    test_size=____,
    random_state=SEED
)

def descriptor_table(frame):
    # Apply calculate_descriptors() to every molecule
    # and organise the results into a DataFrame

    return pd.DataFrame(
        [calculate_descriptors(m) for m in frame["____"]],
        index=frame.index
    )


# Calculate descriptors for training and test molecules
X_train = descriptor_table(____)
X_test = descriptor_table(____)

# Extract experimentally measured solubilities
y_train = train[____]
y_test = test[____]

#Inspect the results

print(f'Training molecules: {len(____)}')
print(f'Test molecules: {len(____)}')
print(f'Blind challenge molecules: {len(____)}')

# Display the first five rows of calculated descriptors
display(X_train.____())




In the SVD workshop, we used a correlation matrix to investigate relationships between different variables. We can use the same approach here to examine how molecular descriptors relate to each other and to experimentally measured aqueous solubility.

The Pearson correlation coefficient (r) describes the strength and direction of a linear relationship between two variables:

r = +1: Perfect positive correlation — both variables increase together.

r = −1: Perfect negative correlation — as one variable increases, the other decreases.

r = 0: No linear correlation.

We will calculate a correlation matrix using .corr() and display it as a heatmap, where colours represent the correlation coefficients.

### Task 5 - Explore molecular propeteries

Complete the code below to combine the molecular descriptors with experimental solubility, calculate the correlation matrix and display it as a heatmap.

In [ ]:

# Copy the molecular descriptors and add experimental solubility
combined_data = ____.copy()
combined_data["Solubility"] = ____

# Calculate the Pearson correlation matrix using pandas
corr_matrix = combined_data.____()

# Display the correlations as a heatmap
plt.figure(figsize=(9, 7))

sns.heatmap(
    ____,
    annot=____,       # Show correlation coefficients
    cmap="____",      # Colour scheme
    vmin=____,        # Minimum correlation (-1)
    vmax=____,        # Maximum correlation (+1)
    center=0,
    fmt=".2f" # Display coefficients to 2 d.p.
)

plt.title("Correlation between molecular properties and solubility")
plt.tight_layout()
plt.show()


### Task 6 – Establish a baseline and fit linear regression

We will use molecular descriptors to predict aqueous solubility.

A **baseline model** provides a simple comparison by predicting the average training-set solubility for every molecule.

**Linear regression** predicts solubility using a weighted combination of the six descriptors:

$$\widehat{\log S}=b_0+b_1x_1+\cdots+b_6x_6$$

Here, $\widehat{\log S}$ is the prediction, $x_1$ to $x_6$ are descriptors, and the $b$ values are coefficients determined by **fitting** the model to training data.

**Standardisation** puts descriptors on comparable scales by subtracting their training mean and dividing by their standard deviation. A **pipeline** combines standardisation and regression, ensuring that scaling is learned only from training data to prevent **data leakage**.

**Task:** Fit the models and compare their predictions on the test set, print the first 5 predictions.

In [ ]:

# Create a baseline model
# DummyRegressor predicts the mean solubility of the training set
baseline = DummyRegressor(strategy="____")


# First standardise the descriptors, then fit linear regression
linear = make_pipeline(
    ____(),
    ____()
)

# Fit both models using the training data
# .fit() learns from the descriptors (X) and measured solubility (y)
baseline.fit(____, ____)
linear.fit(____, ____)


# .predict() uses the fitted model to make predictions
# Predict the test set using the baseline model
baseline_predictions = baseline.predict(X_test)
linear_predictions = linear.predict(____)

# Display the first five linear predicted log S values




### Task 7 – Measure predictive error

We now need to evaluate how accurately our models predict solubility.

A **residual** is the difference between measured and predicted solubility. We will use three measures of model performance:

- **MAE (mean absolute error):** The average size of the prediction errors.
- **RMSE (root mean square error):** Similar to MAE, but gives greater weight to large errors.
- **R²:** Measures how well the model predicts the data compared with predicting their mean. A value of 1 is perfect, 0 matches that reference, and negative values indicate worse performance.

For MAE and RMSE, **lower values are better**. An error of 1 log S unit corresponds to a tenfold difference in solubility.

We will also plot **measured versus predicted values**. Points on the diagonal line represent perfect predictions. A **residual plot** helps identify systematic errors.

**Task:** Use mean_absolute_error() to calculate the MAE for both models. Then calculate the RMSE and R² for linear regression using mean_squared_error(), np.sqrt() and r2_score(). Print the baseline MAE as well as the linear regression MAE, RMSE and R^2

In [ ]:

# Calculate prediction errors for both models
baseline_mae = _______(y_test, ____)
linear_mae = ________(y_test, ____)

linear_rmse = ____.______(_______(y_test, ____))
linear_r2 = r2_score(y_test, ____)

print(f"{____}")
print(f"{____}")
print(f"{____}")
print(f"{____}")


### Task 8 - Fit a nonlinear model and investigate overfitting

Our linear regression model assumes that solubility can be predicted from a linear combination of molecular descriptors. However, relationships between molecular properties and solubility may be more complicated.

A **decision tree regressor** predicts a numerical value by repeatedly dividing molecules into groups based on their descriptors. For example, it might first separate molecules according to their LogP, then divide those groups further based on molecular weight. Each final group is assigned a predicted solubility.

A **random forest regressor** combines many decision trees and averages their predictions. Each tree is trained using a random sample of the training molecules and considers random subsets of descriptors. This helps the model identify **nonlinear relationships** that linear regression may miss.

A **hyperparameter** is a setting chosen before training. We will use:

- `n_estimators`: The number of trees in the forest. We will start with 200.
- `min_samples_leaf`: The minimum number of training molecules in a final group. We will use 3.
- `random_state`: Ensures reproducible results.

**Overfitting** occurs when a model learns patterns specific to the training data, including noise, rather than relationships that generalise to new data. As a result, it performs better on the training set than on unseen test data.

**Your task:** Fit a random forest regressor and compare its MAE on the training and test datasets. Does it predict solubility more accurately than linear regression? Is there evidence of overfitting?

In [ ]:

# Create a random forest regressor
# n_estimators = number of decision trees (use 200)
# min_samples_leaf = minimum molecules per leaf (use 3)
# random_state ensures reproducible results
forest = RandomForestRegressor(
    n_estimators=____,
    min_samples_leaf=____,
    random_state=SEED
)

# Fit the random forest using the training data
forest.fit(____, ____)

# Predict log S for the training and test molecules
forest_train_predictions = forest.predict(____)
forest_test_predictions = forest.predict(____)

# Calculate MAE for the training and test sets
forest_train_mae = mean_absolute_error(y_train, ____)
forest_test_mae = mean_absolute_error(y_test, ____)

# Print the mean absolute errors for the training and test sets to 3 decimal places
print(f"{____}")
print(f"{____}")

# Compare with the linear regression test MAE from Task 6 to 3 decimal places
print(f"Linear regression test MAE: {linear_mae:.3f}")


Compare the random forest's test MAE with that of the linear regression model to determine whether the more complex model improves predictions for unseen molecules. Then compare the random forest's training and test MAE. A substantially lower training MAE suggests that the model may be overfitting, learning patterns specific to the training data rather than relationships that generalise to new molecules.

### Task 9 -Use cross-validation and interpret features

So far, we have evaluated our models using a single split into training and test data. However, the results may depend on which molecules happened to be included in each group.

**Cross-validation (CV)** provides a more reliable way to assess model performance. In **five-fold cross-validation**, the training data are divided into five groups. The model is trained on four groups and evaluated on the remaining group. This is repeated five times, with a different group held out each time. We can then calculate the mean prediction error across the five folds.

We will use cross-validation to evaluate our linear regression and random forest models without using the final test set. Any preprocessing, such as descriptor standardisation, is fitted separately within each training fold.

Next, we will investigate **which molecular descriptors contribute most to the random forest's predictions** using *permutation importance*. This involves randomly shuffling the values of one descriptor at a time in held-out data and measuring how much the prediction error increases. A larger increase in MAE suggests that the model relies more heavily on that descriptor.

Permutation importance does not necessarily tell us which molecular properties directly control solubility. Descriptors may be correlated, and their apparent importance depends on the model and dataset.

**In this section, you will:**
1. Perform five-fold cross-validation to estimate the prediction errors of both models.
2. Compare their average MAE and the variation between folds.
3. Calculate permutation importance for the random forest using the existing test data for this teaching exercise.
4. Identify which molecular descriptors the model relies on most strongly.

Cross-validation can also be used to select model settings (*hyperparameters*). Although we have already examined test-set results for teaching purposes, model selection should normally be completed using validation data before evaluating the final model on an untouched test set.

In the cell below complete the missing code to perform five-fold cross-validation on both the linear regression and random forest models.

Use cross_val_score() with cv=5 to divide the training data into five folds. The function returns negative MAE values, so multiply these by −1 to obtain positive errors.

Compare the mean MAE and standard deviation for the two models. Which model has the lower average prediction error? Does either model show greater variation in performance between folds?

In [ ]:

from sklearn.model_selection import cross_val_score

# Evaluate linear regression using five-fold cross-validation
# Negative MAE is used because sklearn scores models so higher is better
linear_cv = cross_val_score(
    linear, X_train, y_train,
    cv=____,
    scoring="neg_mean_absolute_error"
)

# Repeat for the random forest
forest_cv = cross_val_score(
    ____, X_train, y_train,
    cv=____,
    scoring="neg_mean_absolute_error"
)

# Convert negative scores into positive MAE values.
#  scikit-learn returns negative MAE values during cross-validation because its scoring system treats higher values as better. 
# We multiply these scores by −1 to obtain the usual positive MAE values, where lower values indicate more accurate predictions.
linear_cv_mae = ________
forest_cv_mae = ________

# Print mean MAE and standard deviation across folds
print(f"Linear regression CV MAE: {linear_cv_mae.mean():.3g} ± {linear_cv_mae.std():.3g}")
print(f"Random forest CV MAE: {forest_cv_mae.mean():.3g} ± {forest_cv_mae.std():.3g}")


We will now investigate which molecular descriptors the random forest relies on most strongly when predicting aqueous solubility.

Permutation importance measures the contribution of each descriptor by randomly shuffling its values and observing how much the model's prediction error increases. If shuffling a descriptor causes a large increase in MAE, this suggests that the descriptor is important for the model's predictions.

We will use the random forest trained in Task 8 and the existing test dataset, so there is no need to train another model or split the data again.

Complete the missing code using permutation_importance(). Set n_repeats=10 to repeat the shuffling ten times for each descriptor, and use importance.importances_mean to plot the average increase in MAE.

Examine the resulting bar chart. Which descriptors appear most important for predicting solubility? Are these consistent with your understanding of the molecular properties that influence aqueous solubility?

Remember that permutation importance describes how the model uses the descriptors, not necessarily which molecular properties directly cause changes in solubility. Here, we use the test set for exploratory teaching purposes; in a rigorous study, feature importance would normally be investigated using separate validation data.

In [ ]:
from sklearn.inspection import permutation_importance

# Calculate permutation importance using the existing random forest
# Shuffle each descriptor 10 times and measure the change in MAE
importance = permutation_importance(
    ____,
    ____,
    ____,
    scoring="neg_mean_absolute_error",
    n_repeats=10,
    random_state=SEED
)

# Plot the average increase in MAE for each descriptor as well as the standard deviation
plt.figure(figsize=(8, 4))
plt.bar(X_train.columns, importance._____,
        yerr=importance.______, capsize=4)

plt.xlabel("Molecular descriptor")
plt.ylabel("Increase in MAE")
plt.title("Random forest permutation importance")
plt.xticks(rotation=45)
plt.show()


#### Interpreting cross-validation and feature importance

During cross-validation, descriptor scaling must be fitted separately using the training data in each fold. If scaling is calculated using all the data before cross-validation, information from the validation folds influences the training process. This is known as **data leakage** and can produce overly optimistic estimates of model performance. Using a pipeline ensures that scaling is fitted correctly within each fold.

The descriptor with the highest permutation importance is the one the model relies on most strongly for its predictions. However, this does not necessarily mean that the descriptor directly controls the molecular mechanism of solubility. Correlations between descriptors can also influence their apparent importance.

The **error bars** in a permutation importance plot represent the variation in importance across repeated random shuffles of each descriptor, typically shown as the standard deviation. Larger error bars indicate greater variability in the estimated importance.

### Task 10 - Predict the solubility of ten unknown molecules (25–30 minutes)

We will now use our trained machine-learning approach to predict the aqueous solubility of ten molecules whose experimental solubilities have been withheld from the modelling exercises.

This is a **holdout challenge**: the model must make predictions without using the experimental results for these molecules.

We will use the random forest model with the same settings as before. First, we will train a new random forest using all the available molecules except the ten withheld molecules. This gives the model more training data without revealing the answers.

We will then predict the log S values of the ten molecules and rank them from most soluble to least soluble. Remember that **higher log S values indicate greater aqueous solubility**.

Finally, we can reveal the experimental values and compare them with our predictions to assess how well the model performs on molecules it has never encountered.

In [ ]:

# Create a random forest using the settings from Task 8
final_model = ____

# Fit the model using all non-blind molecules
# Use descriptor_table(remaining) and remaining[TARGET]


# Predict log S for the ten withheld molecules
blind_predictions = ____

# Create a table of molecules and predictions
challenge = blind[['Compound ID', 'smiles']].copy()

# Add a column containing the predicted log S values
challenge['Predicted log S'] = ____

# Convert predicted log S into solubility (mol/L)
challenge['Predicted S (mol/L)'] = ____

# Rank the molecules from most soluble to least soluble
# Use sort_values() with ascending=False
display(____)




### The Grand Reveal!

It's time to uncover the experimental results!

Run the final code cell to reveal the measured solubilities and compare them with your predictions.

Did your model identify the most soluble molecule? How close were your predicted values to the experimental measurements? Were there any surprises?

Calculate the MAE for your ten mystery molecules and compare it with the test MAE from earlier in the workshop.

**The final question:** How much would you trust your model to predict the solubility of a completely new molecule, and what might limit its accuracy?

You've now completed a full molecular machine-learning workflow, from chemical structures to predictions of experimentally measurable properties.

In [ ]:
# THE GRAND REVEAL!

# Reveal the experimental solubilities
challenge['Measured log S'] = [TARGET]

# Convert measured log S into solubility in mol/L
challenge['Measured S (mol/L)'] = 10 ** _________['Measured log S']

# Calculate the residuals (measured minus predicted)
challenge['Residual'] = ____

# Display predictions alongside experimental results
# Rank the molecules from most soluble to least soluble
print(f"THE RESULTS ARE IN!")
display(____)

# Find the highest measured log S value
highest_logS = ____

# Display the most soluble molecule
print(f"The most soluble molecule is:")
display(____)

# Calculate the MAE for the ten mystery molecules
blind_mae = ____

# Display your final score to three significant figures
print(f"Your final prediction MAE: {blind_mae:.3g} log S units")

# Compare with the random forest test MAE from Task 8
print(f"Earlier random forest test MAE: {forest_test_mae:.3g} log S units")

Congratulations! You've built and tested a machine-learning model that predicts a real chemical property from molecular structure.

Over the course of this workshop, you've learned how to:

- Represent molecules: Use SMILES and RDKit to work with chemical structures.

- Calculate molecular descriptors: Convert structures into numerical properties that machine-learning models can use.

- Explore chemical data: Investigate relationships between molecular properties and aqueous solubility.

- Train predictive models: Compare a simple baseline, linear regression and random forest regression.

- Evaluate predictions: Use MAE, RMSE and R² to assess model performance.

- Assess reliability: Use cross-validation to investigate how well models generalise and permutation importance to understand which descriptors the model relies on.

- Predict the unknown: Apply your model to ten previously unseen molecules and compare your predictions with experimental measurements.